# Huấn luyện mô hình Imitation Learning (IL) — Điều khiển bám làn + đèn tín hiệu (CARLA)

**Đồ án:** Nghiên cứu ứng dụng học tăng cường sâu (DRL) điều khiển xe bám làn mô phỏng trong môi trường CARLA.
Notebook này là bước 2/3 — warm-start policy network cho DRL (bước 3, xem `drl_training/`).

**Input (đưa vào model)**: `seg_label` (segmentation, one-hot 4 lớp) + đặc trưng số
(`speed_mps`, `yaw_rate_rps`, `previous_steer`, `previous_longitudinal`, `speed_limit_kmh`,
`traffic_light_state` one-hot 4 lớp).
**Output**: `[steer, longitudinal]` ∈ [-1, 1] (longitudinal âm = phanh, dương = ga).
**Phạm vi**: bám làn **+ xử lý đèn tín hiệu** (dừng khi đỏ) → giữ `traffic_light_state`, 4 nhãn:
`green`, `yellow`, `red`, `unknown`.

> ⚠️ **Hợp đồng quan sát (đã sửa):** `lane_offset_m`, `heading_error_rad`, `is_junction`
> **không** được đưa vào model — chỉ dùng để chẩn đoán ở mục 11 (trả về riêng dưới dạng
> `aux`). Lý do: (1) đây là các đại lượng dùng làm **reward** ở bước DRL, để lọt vào input
> sẽ khiến policy học cách "đọc" trực tiếp sai số thay vì học nhìn ảnh segmentation
> (leakage); (2) DRL observation không có các cột này — nếu IL vẫn dùng thì checkpoint
> warm-start sẽ lệch shape/ngữ nghĩa so với actor DRL. Xem
> `docs/csv_fields_by_task.md` và `docs/manual_thu_thap_du_lieu.md` mục 9.2.

---

## Thay đổi ở v2 — đồng bộ với `train_seg` v4.2

Bản trước lệch với notebook segmentation ở 6 điểm; một trong số đó khiến notebook
**crash ngay batch đầu tiên** (notebook chưa từng có execution_count nào, nên lỗi vẫn còn nguyên).

| # | Bản cũ | v2 |
|---|--------|-----|
| 1 | `SEG_LABEL_LUT[0] = 255` rồi `F.one_hot(..., num_classes=6)`. CARLA 0.9.10 không có tag Sky riêng nên **raw 0 = bầu trời = 26% pixel mọi ảnh** → `RuntimeError: Class values must be smaller than num_classes` ở batch đầu (trên GPU là "device-side assert") | raw 0 → Background, không còn giá trị nào ≥ `NUM_CLASSES`. Thêm chốt `ValueError` có tên file nếu lọt |
| 2 | 6 lớp `Road=0, RoadLine=1, Sidewalk=2, Vegetation=3, Sky=4, Other=5` — **lệch cả số lớp lẫn chỉ số** với seg v4.2 (`Background=0, Road=1, RoadLine=2, Sidewalk=3, Vehicle=4`). Bật `USE_PREDICTED_SEGMENTATION` thì seg trả 1=Road còn IL đọc 1=RoadLine | Dùng chung đúng 5 lớp, đúng thứ tự |
| 3 | `raw 14 Ground → Road`. Ground trong CARLA là bục/vòng xuyến/sân phẳng, **không phải mặt đường** — dạy IL rằng lề bê tông đi được | `Ground → Background`, khớp seg v4.2 |
| 4 | Resize `INTER_NEAREST` 384→192 **trước** khi remap → vạch kẻ (rộng 2–3 px, 1.06% pixel) bị xoá phần lớn | remap trước, rồi `downscale_labels()` bảo tồn lớp mảnh bằng độ phủ diện tích — **cùng một hàm** cho cả ground-truth lẫn mask dự đoán |
| 5 | `USE_PREDICTED_SEGMENTATION` dựng `smp.DeepLabV3Plus` → `load_state_dict` lỗi key vì seg v4.2 là `Unet` + `scse` | Đọc `model_arch`/`encoder_name`/`image_size`/`norm_*` thẳng từ checkpoint, thêm flip-TTA |
| 6 | Bảng nhãn chép tay ở 3 nơi, không ai kiểm | Mục **2b** đối chiếu `class_names` và `label_lut` với file `.pth`, `assert` thẳng nếu lệch |

> ⚠️ **Cần sửa cả bên DRL:** `drl_training/policy/backbone.py` đang gọi `F.one_hot(..., num_classes=6)`.
> Phải đổi thành 5 và dùng đúng `seg_label_lut` + `downscale_labels` lưu trong checkpoint IL,
> nếu không warm-start actor PPO sẽ báo shape mismatch ở conv đầu tiên.

**Giảm mẫu:** `TARGET_TRAIN = 50000`, `TARGET_VAL = 10000` ở cell cấu hình (`None` = dùng hết).
Việc cắt diễn ra **sau** khi chia session nên không tạo rò rỉ mới, và **giữ nguyên 100% frame
hiếm** (đèn vàng/đỏ, `|steer| > 0.1`) — chỉ tỉa nhóm "đi thẳng + đèn xanh". `WeightedRandomSampler`
ở mục 6 chỉ lặp lại frame hiếm chứ không sinh thêm, nên mất một frame cua gấp là mất thật.

---

## Thay đổi ở v3 — bộ dữ liệu 5 FPS, chia theo town

| | v2 | v3 |
|---|-----|-----|
| Dữ liệu | 125k @ 20 FPS, cắt xuống 50k/10k | **40k/10k @ 5 FPS** (10k mỗi town), `TARGET_* = None` — đọc hết, không cắt |
| Không gian nhãn | 5 lớp | **6 lớp khớp `train_seg` v5**: `Background, Road, RoadLine, Sidewalk, Vehicle, Sky`. `raw 0` (bầu trời, 26% pixel) → `Sky` thay vì `ignore` |
| Chia train/val | theo `session_id`, trộn cả 5 town | `SPLIT_MODE = "town"`: Town01–04 train, Town05 val — **giống hệt `train_seg` v5** |
| `previous_steer` | không kiểm tra | mục 4 in `corr(steer, previous_steer)` và MAE của **baseline sao chép**; mục 11 so trực tiếp và cảnh báo nếu model thua |
| Checkpoint | — | thêm `collect_fps`, `control_dt`, `split_mode`, `copycat_*_mae` |

**Vì sao 5 FPS là thay đổi quan trọng nhất ở đây.** `previous_steer` là đặc trưng nguy hiểm
nhất trong behavior cloning: ở 20 FPS vô-lăng gần như không kịp đổi trong 50 ms, nên
`previous_steer ≈ steer` và model đạt loss rất thấp bằng cách **chép lại nó, bỏ qua hoàn toàn
ảnh segmentation**. Lúc chạy thật, `previous_steer` là hành động của chính nó ở bước trước —
sai số tích luỹ và xe trôi khỏi làn mà không có gì kéo lại. Đây là lỗi kinh điển (causal
confusion / copycat problem), và nó **không hiện ra trong val loss**. Ở 200 ms mối tương quan
yếu đi rõ rệt; hai cell kiểm tra mới sẽ cho bạn thấy con số cụ thể.

> **Chia theo town đổi ý nghĩa của val.** Town05 chưa từng xuất hiện lúc train nên MAE sẽ
> **cao hơn** cách chia theo session — đó là con số thật, không phải hồi quy. Đổi lại nó
> đo đúng thứ cần đo trước khi sang DRL, và khớp với cách seg được đánh giá.

> ⚠️ **Hợp đồng với DRL:** đặt `fixed_delta_seconds` của CARLA lúc train DRL đúng bằng
> `control_dt = 0.2 s`. Lệch bước thời gian thì `previous_*` sẽ lệch thang đo ngay bước đầu.

> ⚠️ **Bên DRL:** `in_channels` phải bằng `NUM_CLASSES` của bản này. Nếu số kênh TRÙNG
> nhau mà bảng nhãn khác nhau thì `load_state_dict` **không báo lỗi gì** — model chỉ nhận
> sai kênh và lái sai. Luôn đối chiếu `label_lut` trong checkpoint, đừng tin số kênh.

---

## Thay đổi ở v6 — bỏ `Vehicle`, gộp bầu trời vào `Background`

| | v4/v5 | v6 |
|---|-----|-----|
| Không gian nhãn | 6 lớp `Background, Road, RoadLine, Sidewalk, Vehicle, Sky` | **4 lớp `Background, Road, RoadLine, Sidewalk`** |
| `raw 10` Vehicles | lớp `Vehicle` riêng | → `Background` |
| `raw 0` / `raw 13` (bầu trời) | lớp `Sky` riêng | → `Background` (vẫn được giám sát đầy đủ, **không** phải `ignore`) |
| `in_channels` của `SteeringNet` | 6 | **4** |

**Vì sao bỏ `Vehicle`.** Dataset hiện tại thu ở chế độ không spawn NPC nên `raw 10` gần
như 0% pixel. Với IL, một kênh one-hot toàn số 0 không mang thông tin nhưng vẫn là một
kênh conv thật: nó tiêu tốn tham số ở lớp đầu, và tệ hơn, nó tạo ra một **hợp đồng sai**
với DRL — bước 3 sẽ dựng actor với 6 kênh đầu vào cho một class không bao giờ xuất hiện.
Bước tránh vật cản sẽ làm sau: thu lại dữ liệu **có NPC**, thêm class trong `train_seg`,
rồi train lại đồng bộ cả ba bước.

**Vì sao gộp bầu trời vào `Background`.** Trời và tường trả lời cùng một câu hỏi của bộ
điều khiển — "không đi được". Tách chúng ra chỉ thêm một logit cạnh tranh trong
CrossEntropy và làm `mIoU_all` bên seg bị thổi phồng (~26% pixel với IoU ≈ 0.99). Chú ý
đây **khác hẳn** `ignore`: bản v2 từng đặt `SEG_LABEL_LUT[0] = 255` và làm 26% khung hình
mất hoàn toàn ràng buộc.

> Thay đổi này bắt buộc phải áp dụng ĐỒNG THỜI ở `behavior_cloning/carla_seg.ipynb`
> (`TASK_SCHEME = "lane4"`, `SKY_AS_CLASS = False`),
> `data_collection/carla_collector/schema.py` (`RAW_TO_TRAIN_LANE`) và
> `drl_training/policy/backbone.py` (`NUM_CLASSES = 4`). Mục **2b** dưới đây tự đối chiếu
> notebook này với file `.pth` của seg nên không cần tin vào comment.

## 1. Cài đặt & Import

In [ ]:
# !pip install segmentation-models-pytorch albumentations opencv-python-headless torch torchvision pandas tqdm matplotlib scikit-learn

import os
# [v2] CUDA_LAUNCH_BLOCKING=1 đã bị gỡ. Nó được bật để truy lỗi "device-side assert" do
# F.one_hot gặp pixel 255 — lỗi đó nay đã sửa tận gốc ở LUT (xem cell cấu hình), nên giữ
# lại chỉ còn tác dụng làm chậm training. Đặt lại "1" nếu cần debug CUDA lần nữa.

import glob, random, time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 2. Cấu hình

**Chỉnh `DATASET_ROOT` cho khớp máy bạn** — thư mục gốc chứa các folder `Town01_...`, `Town02_...`, ..., `Town05_...` (cùng quy ước với notebook segmentation). Mỗi Town có `il_fields.csv` **riêng**, với `seg_label_path`/`seg_color_path` là đường dẫn **tương đối theo từng Town** (vd. `seg_label/00173383.png`) — không có sẵn file `driving_log.csv` gộp nào để đọc thẳng. Notebook tự gộp cả 5 Town và nối lại thành đường dẫn tuyệt đối ở mục 3.

In [ ]:
def resolve_dataset_root(explicit=None):
    """Tìm thư mục chứa các folder Town0x_* thay vì hard-code đường dẫn.

    Kaggle mount dataset ở /kaggle/input/<slug>/, mà <slug> đổi mỗi lần bạn upload bản
    mới — hard-code đường dẫn nghĩa là mỗi lần thu thập lại phải sửa tay ở hai notebook
    và rất dễ để một bên trỏ vào dataset cũ. Hàm này dò tự động và IN RA nơi nó tìm thấy.
    """
    cands = ([explicit] if explicit else []) + sorted(
        glob.glob("/kaggle/input/*/CARLA_DATA") +
        glob.glob("/kaggle/input/*/*/CARLA_DATA") +
        glob.glob("/kaggle/input/*/*/*/CARLA_DATA") +
        glob.glob("/kaggle/input/*/*/*/*/CARLA_DATA")
    ) + ["./CARLA_DATA", "./carla_dataset"]

    for c in cands:
        if c and glob.glob(os.path.join(c, "Town01*")):
            towns = sorted(os.path.basename(p) for p in glob.glob(os.path.join(c, "Town0*")))
            print(f"DATASET_ROOT = {c}")
            print(f"  {len(towns)} town: {towns}")
            first = os.path.join(c, towns[0])
            print(f"  {towns[0]}/ chứa: {sorted(os.listdir(first))}")
            return c

    raise FileNotFoundError(
        "Không tìm thấy thư mục nào chứa folder 'Town01*'.\n"
        f"Đã thử: {[c for c in cands if c]}\n"
        "Kiểm tra dataset đã được attach chưa, hoặc truyền thẳng đường dẫn vào "
        "resolve_dataset_root('/kaggle/input/<slug>/CARLA_DATA').")


# [v3.1] Bản cũ để "./carla_dataset" — placeholder này sẽ ném FileNotFoundError ngay
# cell §3. Giờ dò tự động, và dùng CHUNG hàm với train_seg v5 nên hai notebook không
# thể trỏ vào hai dataset khác nhau.
DATASET_ROOT = resolve_dataset_root(None)

# === Cấu trúc thư mục thực tế (giống notebook segmentation) ===
#   DATASET_ROOT/
#     Town01_20260806_.../
#         rgb/, seg_color/, seg_label/, il_fields.csv, drl_fields.csv, astar_fields.csv,
#         metadata.json, states.csv, summary.json
#     Town02_.../ ... Town05_.../
#
# il_fields.csv CHỈ nằm trong từng thư mục Town (không có file gộp "driving_log.csv" nào ở
# DATASET_ROOT), và seg_label_path/seg_color_path bên trong là đường dẫn TƯƠNG ĐỐI theo Town
# đó (vd. "seg_label/00173383.png") — mục 3 sẽ gộp cả 5 Town và nối lại thành đường dẫn
# tuyệt đối trước khi đọc bằng cv2.imread.
# [v3] Chia theo TOWN, giống hệt train_seg v5: Town05 chưa từng xuất hiện lúc train nên
# val đo đúng khả năng tổng quát hoá sang bản đồ mới, và khớp với cách seg được đánh
# giá. (Cách cũ trộn cả 5 town rồi tách 15% session thì lỏng hơn, và khiến IL được val
# trên chính các town mà seg đã nhìn thấy — hai model không còn so sánh được.)
TRAIN_TOWNS = ["Town01", "Town02", "Town03", "Town04"]
VAL_TOWNS   = ["Town05"]
TOWNS       = TRAIN_TOWNS + VAL_TOWNS


def find_town_dirs(root, town_prefix):
    """Tìm TẤT CẢ folder Town0x_* (có hậu tố ngày giờ ngẫu nhiên) bên trong root.

    Trả về danh sách (có thể nhiều phần tử) thay vì 1 thư mục duy nhất: khi thu thập dữ
    liệu theo đúng khuyến nghị (nhiều session ngắn/map, xem data_collection/README.md) thì
    một Town có thể có nhiều folder "Town0x_<timestamp>". Trước đây hàm này chỉ trả về
    matches[0] và âm thầm bỏ qua các session còn lại — khiến phần lớn dữ liệu đã thu thập
    không được dùng để train. Nơi gọi hàm này (mục "Nạp dữ liệu") đã sẵn lặp qua toàn bộ
    danh sách thư mục nên chỉ cần đổi hàm là gộp được hết, không cần sửa gì thêm.
    """
    matches = sorted(glob.glob(os.path.join(root, f"{town_prefix}_*")))
    if not matches:
        exact = os.path.join(root, town_prefix)  # phòng khi folder không có hậu tố
        if os.path.isdir(exact):
            return [exact]
        raise FileNotFoundError(
            f"Không tìm thấy folder cho '{town_prefix}' trong '{root}'. "
            f"Kiểm tra lại DATASET_ROOT."
        )
    return matches


TOWN_DIRS = [d for t in TOWNS for d in find_town_dirs(DATASET_ROOT, t)]
print("Town dirs:")
for d in TOWN_DIRS:
    print("  -", d)

# ============================================================
# [v2] KHÔNG GIAN NHÃN — phải khớp TUYỆT ĐỐI với train_seg v4.2
# ============================================================
# Bản cũ dùng 6 lớp riêng (Road=0, RoadLine=1, Sidewalk=2, Vegetation=3, Sky=4, Other=5).
# Segmentation v4.2 xuất 5 lớp theo THỨ TỰ KHÁC: Background=0, Road=1, RoadLine=2,
# Sidewalk=3, Vehicle=4. Không chỉ lệch số lớp mà lệch cả CHỈ SỐ: nếu bật
# USE_PREDICTED_SEGMENTATION, model seg trả 1=Road nhưng IL đọc 1=RoadLine — toàn bộ mặt
# đường bị hiểu thành vạch kẻ. Giờ hai bên dùng chung một bảng.
# [v6] Khớp train_seg v6 (TASK_SCHEME="lane4", SKY_AS_CLASS=False):
#   - Vehicle bị GỠ: dataset chưa có NPC nên raw 10 ~ 0% pixel. Một kênh one-hot toàn 0
#     không cho model thêm thông tin, nhưng vẫn tạo hợp đồng sai với DRL (actor sẽ dựng
#     thừa một kênh conv cho class không bao giờ xuất hiện). Bước tránh vật cản làm sau.
#   - Sky GỘP vào Background: trời và tường cùng trả lời "không đi được". Đây KHÔNG phải
#     ignore — bầu trời vẫn được giám sát, chỉ là không chiếm một class riêng.
# Thứ tự và tên phải khớp TUYỆT ĐỐI với `class_names` trong checkpoint seg (mục 2b assert).
CLASS_NAMES = ["Background", "Road", "RoadLine", "Sidewalk"]
NUM_CLASSES = len(CLASS_NAMES)
ROADLINE_ID = CLASS_NAMES.index("RoadLine")
ROAD_ID     = CLASS_NAMES.index("Road")
# None khi bầu trời không còn là class riêng; checkpoint vẫn ghi lại trường này để phía
# DRL/demo biết bảng nhãn nào đã được dùng.
SKY_ID      = CLASS_NAMES.index("Sky") if "Sky" in CLASS_NAMES else None

# --- Độ phân giải ------------------------------------------------------------
# Dữ liệu thu thập THỰC TẾ là 480x384, không phải 240x192 như comment cũ ghi: probe §2
# của train_seg in ra `shape mask: [(384, 480)]` khi đọc thẳng file gốc.
SEG_NATIVE_HEIGHT = 384
SEG_NATIVE_WIDTH  = 480
# IL hạ xuống 1/2 cho rẻ (DRL chạy realtime). AdaptiveAvgPool2d trong SteeringNet khiến
# kích thước này KHÔNG ảnh hưởng shape checkpoint — đặt 384/480 nếu muốn bỏ hẳn resize.
IMAGE_HEIGHT = 192
IMAGE_WIDTH  = 240

# --- Hạ mẫu có bảo tồn lớp mảnh ----------------------------------------------
# INTER_NEAREST thuần khi 384->192 XOÁ MẤT vạch kẻ đường: RoadLine chỉ rộng 2-3 px và chỉ
# chiếm 1.06% pixel. Đây đúng là lỗi mà train_seg v4.1 (note #6) đã sửa; bản IL cũ vẫn
# dùng nearest thuần nên đang vứt đi chính tín hiệu quan trọng nhất cho bám làn.
# Hàm này chạy trên TRAIN ID nên dùng được cho CẢ ground-truth LẪN mask dự đoán — bắt
# buộc phải là cùng một hàm, nếu không phân phối lúc train và lúc inference sẽ lệch.
THIN_COVER_THRESH = 0.25

def downscale_labels(lab, out_w=None, out_h=None, thin_ids=None, thr=THIN_COVER_THRESH):
    out_w = IMAGE_WIDTH if out_w is None else out_w
    out_h = IMAGE_HEIGHT if out_h is None else out_h
    if lab.shape[0] == out_h and lab.shape[1] == out_w:
        return lab
    thin_ids = (ROADLINE_ID,) if thin_ids is None else thin_ids
    out = cv2.resize(lab, (out_w, out_h), interpolation=cv2.INTER_NEAREST)
    for t in thin_ids:
        m = (lab == t)
        if not m.any():
            continue
        cov = cv2.resize(m.astype(np.float32), (out_w, out_h), interpolation=cv2.INTER_AREA)
        out[cov > thr] = t
    return out

EPISODE_COL = "session_id"

# --- Bộ dữ liệu 5 FPS --------------------------------------------------------
# [v3] Thu thập lại ở 5 FPS, đúng 40k train / 10k val -> đọc hết, không giảm mẫu.
COLLECT_FPS = 5.0
CONTROL_DT  = 1.0 / COLLECT_FPS     # 0.2 s giữa hai frame liên tiếp
EXPECT_TRAIN, EXPECT_VAL = 40000, 10000   # Town01-04 x 10k  |  Town05 x 10k

# ⚠️ CONTROL_DT là hợp đồng với DRL, không chỉ là metadata.
# `previous_steer` / `previous_longitudinal` nghĩa là "lệnh điều khiển của 1 bước TRƯỚC".
# Ở 20 FPS bước đó cách 50 ms, ở 5 FPS cách 200 ms — cùng một cái tên nhưng khác hẳn ý
# nghĩa. Nếu môi trường DRL step ở tần số khác lúc thu thập, actor warm-start sẽ nhận một
# đặc trưng lệch thang đo ngay từ bước đầu. Đặt fixed_delta_seconds của CARLA lúc train
# DRL đúng bằng CONTROL_DT.

# === Hop dong quan sat (PHAI khop voi DRL - xem `drl_training/policy/observation.py`) ===
# Day la buoc 2/3, warm-start cho DRL (buoc 3). KHONG dua lane_offset_m / heading_error_rad /
# is_junction vao input model: day la cac dai luong dung de tinh REWARD o buoc DRL, khong
# phai observation (xem docs/csv_fields_by_task.md va
# docs/manual_thu_thap_du_lieu.md muc 9.2 "Khong dua lane_offset_m va heading_error_rad
# vao policy observation"). Neu de lot vao observation: (1) model hoc cach "doc" truc tiep
# sai so thay vi hoc nhin anh segmentation - ro ri nhan (leakage); (2) actor DRL warm-start
# tu checkpoint nay se khong tuong thich vi DRL khong co cac cot nay trong input.
CONTINUOUS_COLS = ["speed_mps", "yaw_rate_rps", "speed_limit_kmh"]
RAW_ACTION_COLS = ["previous_steer", "previous_longitudinal"]   # da trong [-1,1], giu nguyen
# lane_offset_m / heading_error_rad / is_junction CHỈ dùng để chẩn đoán ở mục 11 —
# Dataset trả về riêng dưới dạng "aux", không lẫn vào scalar input của model.

# Vocab CO DINH - khong suy ra tu data de tranh lech giua cac lan chay / giua train-val
TRAFFIC_LIGHT_VOCAB = ["green", "yellow", "red", "unknown"]

def normalize_traffic_light(value) -> str:
    v = str(value).strip().lower()
    return v if v in TRAFFIC_LIGHT_VOCAB else "unknown"

SCALAR_FEATURE_DIM = len(CONTINUOUS_COLS) + len(RAW_ACTION_COLS) + len(TRAFFIC_LIGHT_VOCAB)
# Thu tu ghep vector -> phia DRL (`policy/observation.py`) phai build DUNG thu tu nay.
SCALAR_FEATURE_ORDER = list(CONTINUOUS_COLS) + list(RAW_ACTION_COLS) + \
    ["traffic_light_%s" % v for v in TRAFFIC_LIGHT_VOCAB]

BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
USE_AMP = torch.cuda.is_available()
NUM_WORKERS = 4
PERSISTENT_WORKERS = NUM_WORKERS > 0
EARLY_STOP_PATIENCE = 8

STEER_LOSS_WEIGHT = 2.0
LONGITUDINAL_LOSS_WEIGHT = 1.0
HUBER_BETA = 0.15  # SmoothL1: MSE khi |err|<beta, MAE ngoai do -> ben hon voi cac khung lai
                    # gap/hoi phuc lan (kieu DAgger) hiem gap nhung sai so lon.

# Oversample cac session/frame co den vang/do (hiem hon nhieu so voi xanh/unknown)
USE_TRAFFIC_LIGHT_OVERSAMPLING = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
STEER_CHECKPOINT_PATH = "best_il_model.pth"

# Nang cao: True neu muon train tren segmentation DU DOAN (tu model notebook 1) thay vi
# ground-truth - khop dung phan phoi luc inference that. Dataset o muc 3 luon tu suy cot
# "rgb_path" tu "frame" nen khong can chinh gi them de bat co nay.
USE_PREDICTED_SEGMENTATION = False
SEG_CHECKPOINT_PATH = "best_carla_lane_seg.pth"   # [v2] tên checkpoint của train_seg v4.2
PREDICTED_MASK_DIR = "./predicted_seg_masks"

# Palette 4 class — copy nguyên COLORS của train_seg v6, đúng thứ tự CLASS_NAMES.
PALETTE = np.array([
    [ 60,  60,  60],  # 0 Background (bầu trời, nhà, cây, cột, hàng rào, xe, Ground...)
    [128,  64, 128],  # 1 Road
    [157, 234,  50],  # 2 RoadLine
    [244,  35, 232],  # 3 Sidewalk
], dtype=np.uint8)

# ============================================================
# LUT raw CARLA (0-22) -> 4 train id. PHẢI khớp byte-for-byte với LABEL_LUT của
# train_seg v6. Mục 2b đối chiếu tự động với file .pth nên không cần tin vào comment.
# ============================================================
# Chỉ 4 raw id được nhắc tên; MỌI raw id còn lại rơi về Background (0). Các quyết định
# đáng chú ý, tất cả đều là lỗi đã từng mắc:
#   raw 0  Unlabeled : cũ -> 255 (ignore). CARLA 0.9.10 không có tag Sky riêng nên toàn
#                      bộ bầu trời nằm ở raw 0 = 26% pixel MỌI ảnh, và F.one_hot gặp 255
#                      là RuntimeError ngay batch đầu. v4 -> class Sky riêng.
#                      [v6] Nay -> Background: vẫn giám sát đầy đủ, nhưng bầu trời không
#                      đáng một class riêng khi nó trả lời cùng câu hỏi với tường/cây.
#   raw 10 Vehicles  : cũ -> class Vehicle. [v6] Nay -> Background (dataset chưa có NPC;
#                      xem ghi chú v6 ở đầu notebook, bước tránh vật cản làm sau).
#   raw 14 Ground    : cũ -> Road. Ground trong CARLA là bục/vòng xuyến/sân phẳng, KHÔNG
#                      phải mặt đường. Dạy IL rằng lề bê tông đi được là sai nguy hiểm
#                      nhất có thể có với bám làn. -> Background.
#   raw 22 Terrain   : cũ -> Vegetation (class riêng). -> Background, seg không tách
#                      Vegetation nữa.
RAW_TO_TRAIN = {
     6: 2,   # RoadLine  -> RoadLine
     7: 1,   # Road      -> Road
     8: 3,   # Sidewalk  -> Sidewalk
    16: 1,   # RailTrack -> Road  (RAILTRACK_AS_ROAD=True bên seg)
}
# Mọi raw id còn lại (bầu trời raw 0/13, Vehicles 10, Pedestrian 4, Building, Fence, Pole,
# Vegetation, Ground, Terrain, id lạ > 22...) -> Background
SEG_LABEL_LUT = np.zeros(256, dtype=np.uint8)
for _raw, _train in RAW_TO_TRAIN.items():
    SEG_LABEL_LUT[_raw] = _train
del _raw, _train

# Mask dự đoán đã là train id sẵn -> KHÔNG áp LUT lần hai (xem Dataset ở mục 6).
SEG_PATHS_ARE_TRAIN_IDS = False


def seed_worker(worker_id):
    """Reseed `random`/numpy rieng cho tung DataLoader worker.

    Khong co ham nay thi torch chi tu seed lai RNG cua chinh no cho moi worker - RNG cua
    `random`/`numpy` bi nhan ban GIONG HET nhau giua cac worker. Nghiem trong nhat tren
    Windows/macOS (multiprocessing start method = spawn): moi worker import lai toan bo
    module va chay lai `random.seed(SEED)` o dau file => TAT CA worker sinh cung mot chuoi
    quyet dinh flip/augment. Ket qua la augmentation kem da dang hon nhieu so voi tuong.
    """
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


DATALOADER_GENERATOR = torch.Generator()
DATALOADER_GENERATOR.manual_seed(SEED)

## 2b. Kiểm tra hợp đồng nhãn với model segmentation

In [ ]:
# ============================================================
# [v2] KIỂM TRA HỢP ĐỒNG NHÃN VỚI train_seg v4.2 — chạy trước khi train
# ============================================================
# Đây là chốt chặn quan trọng nhất của notebook này. Hai model phải nói cùng một "ngôn ngữ
# lớp"; trước đây bảng nhãn được CHÉP TAY ở ba nơi (notebook seg, notebook IL,
# data_collection/carla_collector/schema.py) và đã lệch nhau trong thực tế. Cell này đối
# chiếu tự động với chính file .pth thay vì tin vào comment.
if os.path.exists(SEG_CHECKPOINT_PATH):
    _sc = torch.load(SEG_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
    _seg_names = list(_sc["class_names"])
    _seg_lut   = np.array(_sc["label_lut"], dtype=np.int64)

    print(f"seg checkpoint : {SEG_CHECKPOINT_PATH}")
    print(f"  lớp          : {_seg_names}")
    print(f"  kích thước   : {_sc.get('image_height')}x{_sc.get('image_width')}")
    print(f"  ignore_index : {_sc.get('ignore_index')}")
    print(f"  lane_mIoU    : {_sc.get('best_lane_miou', float('nan')):.4f}")

    assert _seg_names == CLASS_NAMES, (
        f"LỆCH TÊN/THỨ TỰ LỚP.\n  seg: {_seg_names}\n  IL : {CLASS_NAMES}\n"
        "Sửa CLASS_NAMES ở cell cấu hình cho khớp seg, ĐỪNG sửa ngược lại.")

    _diff = [(r, int(_seg_lut[r]), int(SEG_LABEL_LUT[r])) for r in range(23)
             if int(_seg_lut[r]) != int(SEG_LABEL_LUT[r])]
    assert not _diff, ("LỆCH LUT tại (raw, seg, IL): " + str(_diff) +
                       "\nThường gặp: seg đang để IGNORE_UNLABELED=True nên raw 0 -> 255.")

    if int(_seg_lut[0]) == 255:
        raise RuntimeError(
            "seg checkpoint map raw 0 (bầu trời, 26% pixel) sang ignore_index. "
            "Train lại seg với IGNORE_UNLABELED=False trước khi train IL.")

    # [v6] Bắt đúng hai thay đổi của phiên bản này, với thông báo nói rõ phải sửa ở đâu —
    # assert tên lớp ở trên đã đủ để chặn, nhưng nó chỉ in ra hai danh sách lệch nhau.
    if "Vehicle" in _seg_names:
        raise RuntimeError(
            "seg checkpoint còn class 'Vehicle' nhưng IL v6 đã bỏ (dataset chưa có NPC). "
            "Train lại seg với TASK_SCHEME='lane4', hoặc nếu bạn ĐÃ thu dữ liệu có NPC "
            "thì thêm 'Vehicle' lại vào CLASS_NAMES/RAW_TO_TRAIN ở đây và cập nhật cả "
            "data_collection/carla_collector/schema.py + drl_training/policy/backbone.py.")

    # [v4] Đối chiếu luôn bước thời gian. Seg không dùng tới nó, nhưng nếu hai notebook
    # đọc hai bản dataset khác nhau thì đây là chỗ lộ ra sớm nhất.
    _fps = _sc.get("collect_fps")
    if _fps is not None and abs(_fps - COLLECT_FPS) > 1e-6:
        raise RuntimeError(f"LỆCH FPS: seg={_fps}, IL={COLLECT_FPS}. "
                           "Hai notebook đang đọc hai bản dataset khác nhau.")
    if bool(_sc.get("sky_as_class")) != ("Sky" in CLASS_NAMES):
        raise RuntimeError(
            "Lệch cách xử lý bầu trời: seg sky_as_class=%s còn IL %s class 'Sky'. "
            "v6 đặt SKY_AS_CLASS=False ở cả hai (bầu trời gộp vào Background)."
            % (_sc.get("sky_as_class"), "có" if "Sky" in CLASS_NAMES else "không có"))

    print("\n✅ Hợp đồng nhãn khớp: LUT, tên lớp, thứ tự index và FPS đều giống nhau.")
    del _sc, _seg_names, _seg_lut, _diff
else:
    print(f"⚠️  Không thấy {SEG_CHECKPOINT_PATH} — bỏ qua kiểm tra hợp đồng.")
    print("   Train bằng ground-truth vẫn chạy được, nhưng hãy add checkpoint seg làm")
    print("   input dataset để cell này xác nhận hai model dùng chung bảng nhãn.")
    print(f"   Bảng IL đang dùng: {CLASS_NAMES}")

# Sổ tay đối chiếu nhanh, in ra để dán vào docs/csv_fields_by_task.md
print("\nraw CARLA -> train id:")
for _r in range(23):
    _t = int(SEG_LABEL_LUT[_r])
    print(f"  raw {_r:>2} -> {_t} {CLASS_NAMES[_t]}")

## 3. Nạp dữ liệu

In [ ]:
def load_town_il_csv(town_dir, town=None):
    """Đọc il_fields.csv của 1 Town và quy đổi các cột đường dẫn ảnh thành đường dẫn
    tuyệt đối — trong CSV chúng vốn tương đối theo thư mục Town (vd.
    "seg_label/00173383.png"), chỉ đọc đúng nếu cwd trùng thư mục đó. Nối với `town_dir`
    để đọc được bất kể notebook chạy từ đâu, và bất kể đang gộp nhiều Town cùng DataFrame."""
    csv_path = os.path.join(town_dir, "il_fields.csv")
    town_df = pd.read_csv(csv_path)
    town_df["seg_label_path"] = town_df["seg_label_path"].apply(lambda p: os.path.join(town_dir, p))
    # il_fields.csv không có cột rgb_path (không cần cho train IL thường), nhưng
    # USE_PREDICTED_SEGMENTATION cần nó — suy trực tiếp từ "frame", cùng quy ước đặt tên
    # file 8 chữ số như seg_label_path/seg_color_path (vd. frame=173383 -> "rgb/00173383.png").
    town_df["rgb_path"] = town_df["frame"].apply(lambda f: os.path.join(town_dir, "rgb", f"{int(f):08d}.png"))
    # [v3] Giữ lại tên town để chia train/val theo bản đồ, giống train_seg v5.
    town_df["town"] = town if town is not None else os.path.basename(town_dir).split("_")[0]
    return town_df


df = pd.concat([load_town_il_csv(d, t) for t in TOWNS for d in find_town_dirs(DATASET_ROOT, t)],
               ignore_index=True)

required = ["session_id","seg_label_path","speed_mps","yaw_rate_rps","previous_steer",
            "previous_longitudinal","lane_offset_m","heading_error_rad","speed_limit_kmh",
            "traffic_light_state","is_junction","steer","longitudinal"]
missing = [c for c in required if c not in df.columns]
assert not missing, f"CSV thiếu cột: {missing}. Cột hiện có: {list(df.columns)}"

df["traffic_light_state"] = df["traffic_light_state"].apply(normalize_traffic_light)

print(f"Tổng {len(df)} mẫu, {df['session_id'].nunique()} session, "
      f"{len(df)/COLLECT_FPS/60:.0f} phút lái ở {COLLECT_FPS:.0f} FPS")
print(df.groupby("town").size().to_string())
print(df["traffic_light_state"].value_counts())
df.head()

### (Tuỳ chọn nâng cao) Dùng segmentation dự đoán thay ground-truth

Chỉ chạy nếu `USE_PREDICTED_SEGMENTATION = True` và CSV có cột `rgb_path`.

In [ ]:
if USE_PREDICTED_SEGMENTATION:
    assert "rgb_path" in df.columns, "Cần cột 'rgb_path' trong CSV."
    os.makedirs(PREDICTED_MASK_DIR, exist_ok=True)
    _ck = torch.load(SEG_CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)

    # [v2] Bản cũ dựng smp.DeepLabV3Plus -> load_state_dict báo lỗi thiếu/thừa key vì
    # seg v4.2 là Unet + decoder_attention_type="scse". Giờ đọc thẳng kiến trúc từ
    # checkpoint thay vì chép tay.
    _arch = _ck.get("model_arch", "unet")
    _enc  = _ck.get("encoder_name", "resnet34")
    if _arch == "unet":
        seg_model = smp.Unet(encoder_name=_enc, encoder_weights=None,
                             decoder_attention_type="scse", in_channels=3,
                             classes=_ck["num_classes"])
    else:
        seg_model = smp.DeepLabV3Plus(encoder_name=_enc, encoder_weights=None,
                                      encoder_output_stride=8, in_channels=3,
                                      classes=_ck["num_classes"])
    seg_model.load_state_dict(_ck["model_state_dict"])
    seg_model.to(DEVICE).eval()
    for p in seg_model.parameters():
        p.requires_grad = False

    # Suy luận ở ĐÚNG độ phân giải seg được train (384x480) rồi mới hạ xuống kích thước
    # IL bằng downscale_labels — giống hệt đường đi của ground-truth ở mục 6.
    _H = _ck.get("image_height", SEG_NATIVE_HEIGHT)
    _W = _ck.get("image_width",  SEG_NATIVE_WIDTH)
    infer_tf = A.Compose([
        A.Resize(height=_H, width=_W, interpolation=cv2.INTER_LINEAR),
        A.Normalize(mean=tuple(_ck.get("norm_mean", (0.485, 0.456, 0.406))),
                    std=tuple(_ck.get("norm_std",  (0.229, 0.224, 0.225)))),
        ToTensorV2(),
    ])
    print(f"seg: {_arch}/{_enc} @ {_H}x{_W}, {_ck['num_classes']} lớp {_ck['class_names']}")

    predicted_paths = []
    with torch.no_grad():
        # Khoá cache bằng "session_id_frame", KHÔNG phải basename(rgb_path): "frame" là số
        # đếm nội bộ từng phiên và các Town có dải trùng nhau -> dùng basename sẽ âm thầm
        # lấy nhầm mask của Town khác cùng số frame.
        for session_id, rgb_path in tqdm(zip(df["session_id"], df["rgb_path"]), total=len(df),
                                          desc="Sinh segmentation dự đoán"):
            cache_key = f"{session_id}_{os.path.basename(rgb_path)}"
            out_path = os.path.join(PREDICTED_MASK_DIR, cache_key.replace(".png", "_pred.png"))
            if not os.path.exists(out_path):
                img = cv2.cvtColor(cv2.imread(rgb_path, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
                t = infer_tf(image=img)["image"].unsqueeze(0).to(DEVICE)
                with torch.amp.autocast("cuda", enabled=USE_AMP):
                    out = seg_model(t) + torch.flip(seg_model(torch.flip(t, dims=[3])), dims=[3])
                cv2.imwrite(out_path, torch.argmax(out, dim=1)[0].to(torch.uint8).cpu().numpy())
            predicted_paths.append(out_path)
    df["seg_label_path"] = predicted_paths
    SEG_PATHS_ARE_TRAIN_IDS = True     # đã là train id -> Dataset sẽ KHÔNG áp LUT nữa
    del seg_model, _ck
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print("Đã chuyển sang segmentation dự đoán (train id 0..%d)." % (NUM_CLASSES - 1))
else:
    SEG_PATHS_ARE_TRAIN_IDS = False
    print("Dùng ground-truth segmentation (raw id CARLA -> LUT).")

## 4. Chia train/val theo **town** + kiểm tra sao chép hành động


In [ ]:
def split_by_town(frame):
    tr = frame[frame["town"].isin(TRAIN_TOWNS)].reset_index(drop=True)
    va = frame[frame["town"].isin(VAL_TOWNS)].reset_index(drop=True)
    assert len(tr) and len(va), \
        f"Chia theo town thất bại — town có mặt: {sorted(frame['town'].unique())}"
    return tr, va


train_df, val_df = split_by_town(df)
print(f"Chia theo TOWN: train={TRAIN_TOWNS} val={VAL_TOWNS}")

for _nm, _got, _exp in (("train", len(train_df), EXPECT_TRAIN),
                        ("val",   len(val_df),   EXPECT_VAL)):
    if abs(_got - _exp) > 0.02 * _exp:
        print(f"  [!] {_nm}: {_got} mẫu, kỳ vọng ~{_exp} ({100*(_got-_exp)/_exp:+.1f}%)")

# Tổng đúng vẫn có thể che một town thiếu + một town thừa -> kiểm tra từng town.
_per = EXPECT_TRAIN / len(TRAIN_TOWNS)
for _t, _n in train_df.groupby("town").size().items():
    print(f"  {_t}: {_n} mẫu (kỳ vọng ~{_per:.0f})"
          + ("   <-- lệch" if abs(_n - _per) > 0.05 * _per else ""))

print(f"\nTrain: {len(train_df)} mẫu ({train_df[EPISODE_COL].nunique()} session) | "
      f"Val: {len(val_df)} mẫu ({val_df[EPISODE_COL].nunique()} session)")
print("\nPhân bố đèn tín hiệu (train):"); print(train_df["traffic_light_state"].value_counts(normalize=True).round(3))

# ============================================================
# [v3] KIỂM TRA "SAO CHÉP HÀNH ĐỘNG" (causal confusion) — lý do chính để thu ở 5 FPS
# ============================================================
# `previous_steer` là đặc trưng nguy hiểm nhất trong behavior cloning. Ở 20 FPS, steer
# gần như không kịp đổi trong 50 ms nên previous_steer ≈ steer: model đạt loss rất thấp
# bằng cách CHÉP LẠI nó và bỏ qua hoàn toàn ảnh segmentation. Lúc chạy thật thì
# previous_steer là hành động của chính nó ở bước trước -> sai số tích luỹ, xe trôi khỏi
# làn mà không có gì kéo lại. Ở 5 FPS (200 ms) mối tương quan này yếu đi rõ rệt, buộc
# model phải nhìn đường.
#
# Con số dưới đây là NGƯỠNG PHẢI VƯỢT ở mục 11. Nếu MAE của model không thấp hơn hẳn
# baseline sao chép, model chưa học được gì từ ảnh — dù val_loss trông đẹp thế nào.
for _nm, _d in (("train", train_df), ("val", val_df)):
    _r = _d["steer"].corr(_d["previous_steer"])
    _mae = (_d["steer"] - _d["previous_steer"]).abs().mean()
    print(f"\n[{_nm}] corr(steer, previous_steer) = {_r:.4f}")
    print(f"       MAE của baseline 'steer = previous_steer' = {_mae:.4f}")
COPYCAT_STEER_MAE = float((val_df["steer"] - val_df["previous_steer"]).abs().mean())
COPYCAT_LONG_MAE  = float((val_df["longitudinal"] - val_df["previous_longitudinal"]).abs().mean())
print(f"\nNgưỡng cần vượt ở mục 11: steer < {COPYCAT_STEER_MAE:.4f}, "
      f"longitudinal < {COPYCAT_LONG_MAE:.4f}")
print("corr > 0.98 nghĩa là bước thời gian vẫn quá dày — kiểm tra lại FPS lúc thu thập.")


## 5. Chuẩn hoá đặc trưng số

Z-score fit **chỉ trên train** để tránh rò rỉ dữ liệu.

In [ ]:
def compute_norm_stats(df, cols):
    return {c: (float(df[c].mean()), float(df[c].std()) + 1e-6) for c in cols}

norm_stats = compute_norm_stats(train_df, CONTINUOUS_COLS)
for c, (m, s) in norm_stats.items():
    print(f"{c}: mean={m:.4f} std={s:.4f}")


## 6. Dataset

One-hot segmentation; lật ngang kèm đảo dấu đồng bộ (`steer`, `previous_steer`, `yaw_rate_rps`, `lane_offset_m`, `heading_error_rad`).

In [ ]:
class SteeringDataset(Dataset):
    def __init__(self, dataframe, augment):
        self.df = dataframe.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        mask = cv2.imread(row["seg_label_path"], cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(row["seg_label_path"])

        # [v2] THỨ TỰ ĐÚNG: remap -> rồi mới hạ mẫu. Bản cũ resize NEAREST TRƯỚC ở
        # 384->192 nên vạch kẻ đường (rộng 2-3 px, 1.06% pixel) bị xoá phần lớn TRƯỚC khi
        # kịp được gán nhãn RoadLine. downscale_labels giữ lại lớp mảnh bằng độ phủ diện
        # tích, và là CÙNG một hàm dùng cho mask dự đoán -> train và inference không lệch.
        if not SEG_PATHS_ARE_TRAIN_IDS:
            mask = SEG_LABEL_LUT[mask]      # raw CARLA (0-22) -> train id (0-4)
        mask = downscale_labels(mask)

        # Chốt an toàn: pixel >= NUM_CLASSES sẽ làm F.one_hot ném RuntimeError trên CPU và
        # "device-side assert" trên GPU (đúng lỗi mà bản cũ mắc phải với raw 0 -> 255).
        if mask.max() >= NUM_CLASSES:
            raise ValueError(
                f"train id {mask.max()} >= NUM_CLASSES={NUM_CLASSES} tại "
                f"{row['seg_label_path']} — LUT hoặc checkpoint seg không khớp.")

        steer = float(row["steer"]); longitudinal = float(row["longitudinal"])
        previous_steer = float(row["previous_steer"])
        yaw_rate = float(row["yaw_rate_rps"])
        lane_offset = float(row["lane_offset_m"])        # chi dung cho aux (danh gia)
        heading_error = float(row["heading_error_rad"])  # chi dung cho aux (danh gia)

        if self.augment and random.random() < 0.5:
            mask = np.ascontiguousarray(np.fliplr(mask))
            steer, previous_steer = -steer, -previous_steer
            yaw_rate = -yaw_rate
            lane_offset, heading_error = -lane_offset, -heading_error

        mask_tensor = F.one_hot(torch.from_numpy(mask.astype(np.int64)), num_classes=NUM_CLASSES).permute(2, 0, 1).float()

        cont_src = {"speed_mps": float(row["speed_mps"]), "yaw_rate_rps": yaw_rate,
                    "speed_limit_kmh": float(row["speed_limit_kmh"])}
        cont_vals = [(cont_src[c] - norm_stats[c][0]) / norm_stats[c][1] for c in CONTINUOUS_COLS]
        raw_vals = [previous_steer, float(row["previous_longitudinal"])]
        tl_onehot = [1.0 if row["traffic_light_state"] == v else 0.0 for v in TRAFFIC_LIGHT_VOCAB]
        scalar = np.array(cont_vals + raw_vals + tl_onehot, dtype=np.float32)

        # aux: CHI de chan doan o muc 11, khong dua vao model (xem giai thich o cell cau hinh).
        aux = np.array([lane_offset, heading_error, float(row["is_junction"])], dtype=np.float32)

        target = torch.tensor([steer, longitudinal], dtype=torch.float32)
        return mask_tensor, torch.from_numpy(scalar), torch.from_numpy(aux), target


train_ds = SteeringDataset(train_df, augment=True)
val_ds = SteeringDataset(val_df, augment=False)

# worker_init_fn + generator: xem ham seed_worker() o cell cau hinh - tranh cac DataLoader
# worker sinh cung mot chuoi augmentation (bug pho bien, nghiem trong hon tren Windows/spawn).
_loader_kwargs = dict(
    num_workers=NUM_WORKERS, pin_memory=True, generator=DATALOADER_GENERATOR,
    worker_init_fn=seed_worker if NUM_WORKERS > 0 else None,
    persistent_workers=PERSISTENT_WORKERS,
)
if NUM_WORKERS > 0:
    _loader_kwargs["prefetch_factor"] = 4

if USE_TRAFFIC_LIGHT_OVERSAMPLING:
    tl_freq = train_df["traffic_light_state"].value_counts(normalize=True)
    tl_weight = train_df["traffic_light_state"].map(lambda v: 1.0 / tl_freq[v]).values

    # Can bang them theo do lon |steer|: phan lon mau la "di thang" (duong thang
    # dai, dung den do/ket xe) -> neu khong can bang, batch se bi ap dao boi cac
    # mau gan nhu trung lap ve mat hanh dong, mo hinh de hoc lech ve "giu nguyen
    # vo-lang". KHONG xoa mau nao (khac voi dedup o thu thap/build_manifest, cai
    # nay chi bo cac frame dung yen thuc su trung lap) - van giu nguyen moi mau
    # recovery/cua gap hiem gap, chi giam TAN SUAT xuat hien cua nhom da so trong
    # moi epoch. Bins theo |steer|: gan nhu thang / lai nhe / cua vua / cua gap.
    steer_bins = pd.cut(train_df["steer"].abs(), bins=[-0.001, 0.02, 0.1, 0.3, np.inf],
                         labels=["straight", "gentle", "moderate", "sharp"])
    steer_freq = steer_bins.value_counts(normalize=True)
    steer_weight = steer_bins.map(lambda b: 1.0 / steer_freq[b]).values

    sample_weights = tl_weight * steer_weight
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_df), replacement=True,
                                     generator=DATALOADER_GENERATOR)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, drop_last=True, **_loader_kwargs)
    print("Phân bố bin |steer| (train):"); print(steer_bins.value_counts(normalize=True).round(3))
    print("Dùng WeightedRandomSampler — oversample nhãn đèn hiếm (yellow/red) + cân bằng theo mức steer.")
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, **_loader_kwargs)

val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **_loader_kwargs)

## 7. Trực quan hóa dữ liệu mẫu

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i in range(3):
    m, s, aux, t = train_ds[random.randint(0, len(train_ds) - 1)]
    lab = torch.argmax(m, dim=0).numpy()
    axes[i].imshow(PALETTE[lab])
    # % RoadLine để kiểm tra mắt thường rằng vạch kẻ SỐNG SÓT qua bước hạ mẫu 384->192.
    axes[i].set_title(f"steer={t[0]:.2f} long={t[1]:.2f} | RoadLine {100*(lab==ROADLINE_ID).mean():.2f}%")
    axes[i].axis("off")
handles = [plt.Rectangle((0, 0), 1, 1, fc=PALETTE[i] / 255) for i in range(NUM_CLASSES)]
fig.legend(handles, CLASS_NAMES, ncol=NUM_CLASSES, loc="lower center", frameon=False)
plt.tight_layout(); plt.show()
print("RoadLine trong ground-truth gốc chiếm ~1.06% pixel. Nếu ở đây tụt xuống dưới ~0.6%")
print("thì downscale_labels chưa chạy đúng — vạch kẻ đang bị bước hạ mẫu ăn mất.")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df["steer"], bins=50); axes[0].set_title("Phân bố Steer")
axes[1].hist(df["longitudinal"], bins=50); axes[1].set_title("Phân bố Longitudinal")
df["traffic_light_state"].value_counts().reindex(TRAFFIC_LIGHT_VOCAB).plot(kind="bar", ax=axes[2])
axes[2].set_title("Phân bố Traffic Light State")
plt.tight_layout(); plt.show()

## 8. Model — 2 nhánh (CNN + MLP)

In [ ]:
class SteeringNet(nn.Module):
    """CNN (segmentation one-hot) + MLP (scalar), 2 nhánh -> [steer, longitudinal].

    QUAN TRỌNG: kiến trúc và TÊN thuộc tính (`conv`, `pool`, `cnn_fc`, `scalar_mlp`, `head`)
    phải khớp với `drl_training/policy/backbone.py` + `actor_critic.py` — đó là nơi actor PPO
    nạp lại trọng số warm-start từ checkpoint này (`best_il_model.pth`). Đổi kiến trúc ở đây
    thì phải cập nhật đồng bộ bên DRL, nếu không loader sẽ báo lỗi shape mismatch.
    """
    # [v6] in_channels = NUM_CLASSES = 4:
    #   Background=0 / Road=1 / RoadLine=2 / Sidewalk=3.
    # `drl_training/policy/backbone.py` phải đặt NUM_CLASSES = 4 và dùng ĐÚNG bảng LUT
    # này (nguồn sự thật: data_collection/carla_collector/schema.py::RAW_TO_TRAIN_LANE).
    # Nếu chỉ số kênh khớp mà bảng nhãn khác nhau thì sẽ KHÔNG có lỗi nào báo — model
    # đơn giản nhận sai kênh và lái sai. Đây là loại lỗi im lặng, kiểm tra bằng tay.
    def __init__(self, in_channels=NUM_CLASSES, num_scalar_features=SCALAR_FEATURE_DIM):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 24, 5, 2, 2), nn.BatchNorm2d(24), nn.ELU(),
            nn.Conv2d(24, 36, 5, 2, 2), nn.BatchNorm2d(36), nn.ELU(),
            nn.Conv2d(36, 48, 5, 2, 2), nn.BatchNorm2d(48), nn.ELU(),
            nn.Conv2d(48, 64, 3, 2, 1), nn.BatchNorm2d(64), nn.ELU(),
            nn.Conv2d(64, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ELU(),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.cnn_fc = nn.Sequential(nn.Linear(64, 64), nn.ELU())
        self.scalar_mlp = nn.Sequential(nn.Linear(num_scalar_features, 32), nn.ELU(), nn.Linear(32, 32), nn.ELU())
        self.head = nn.Sequential(
            nn.Linear(64 + 32, 64), nn.ELU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(0.2),
            nn.Linear(32, 2),
        )

    def forward(self, seg_map, scalar_features):
        x_img = self.cnn_fc(self.pool(self.conv(seg_map)).flatten(1))
        x_sca = self.scalar_mlp(scalar_features)
        out = self.head(torch.cat([x_img, x_sca], dim=1))
        return torch.tanh(out)  # tanh theo tung phan tu -> tuong duong tach steer/long rieng


def control_loss(pred, target):
    # SmoothL1 (Huber) thay MSE thuan: ben hon voi cac khung lai gap/hoi phuc lan hiem gap
    # nhung sai so lon (kieu DAgger), tranh vai mau ngoai lai chi phoi gradient ca batch.
    steer_l = F.smooth_l1_loss(pred[:, 0], target[:, 0], beta=HUBER_BETA)
    long_l = F.smooth_l1_loss(pred[:, 1], target[:, 1], beta=HUBER_BETA)
    return STEER_LOSS_WEIGHT * steer_l + LONGITUDINAL_LOSS_WEIGHT * long_l


def build_param_groups(module, weight_decay):
    """Tach bias/BatchNorm ra khoi weight decay - thuc hanh chuan (vd. AdamW trong timm/
    torchvision reference training), tranh phat norm layer va bias (khong de overfit,
    bi regularize sai cach se lam giam hieu nang thay vi tang generalization)."""
    decay, no_decay = [], []
    for name, param in module.named_parameters():
        if not param.requires_grad:
            continue
        if param.ndim <= 1 or name.endswith(".bias"):
            no_decay.append(param)
        else:
            decay.append(param)
    return [
        {"params": decay, "weight_decay": weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]


# SteeringNet chỉ ~0.15M tham số — DataParallel đã bị gỡ: chi phí đồng bộ giữa 2 GPU
# lớn hơn phần tính toán tiết kiệm được, và nó thêm tiền tố "module." vào state_dict
# khiến phía DRL phải xử lý thêm một trường hợp.
model = SteeringNet().to(DEVICE)

optimizer = torch.optim.AdamW(build_param_groups(model, WEIGHT_DECAY), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)   # [v2] API cũ đã deprecated
print(f"Số tham số: {sum(p.numel() for p in model.parameters())/1e6:.3f}M")


## 9. Vòng lặp huấn luyện

In [ ]:
def run_epoch(loader, train):
    model.train() if train else model.eval()
    total_loss, total_mae, n = 0.0, np.zeros(2), 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for masks, scalars, _aux, targets in loader:
            masks = masks.to(DEVICE, non_blocking=True)
            scalars = scalars.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            if train: optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                preds = model(masks, scalars)
                loss = control_loss(preds, targets)
            if train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(optimizer); scaler.update()
            total_loss += loss.item()
            total_mae += torch.abs(preds - targets).mean(dim=0).detach().cpu().numpy()
            n += 1
    return total_loss / n, total_mae / n


In [ ]:
history = {"train_loss": [], "val_loss": [], "val_steer_mae": [], "val_long_mae": [], "epoch_time": [], "lr": []}
best_val_loss = float("inf"); epochs_no_improve = 0
training_start = time.time()

pbar = tqdm(range(EPOCHS), desc="Training IL")
for epoch in pbar:
    t0 = time.time()
    train_loss, _ = run_epoch(train_loader, True)
    val_loss, val_mae = run_epoch(val_loader, False)
    scheduler.step()

    history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
    history["val_steer_mae"].append(val_mae[0]); history["val_long_mae"].append(val_mae[1])
    history["epoch_time"].append(time.time() - t0); history["lr"].append(optimizer.param_groups[0]["lr"])
    pbar.set_postfix({"train_loss": f"{train_loss:.4f}", "val_loss": f"{val_loss:.4f}", "steer_mae": f"{val_mae[0]:.4f}"})

    if val_loss < best_val_loss:
        best_val_loss = val_loss; epochs_no_improve = 0
        torch.save({
            "epoch": epoch, "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(), "scheduler_state_dict": scheduler.state_dict(),
            "best_val_loss": best_val_loss, "num_classes": NUM_CLASSES,
            "image_height": IMAGE_HEIGHT, "image_width": IMAGE_WIDTH,
            # [v2] Hợp đồng nhãn đi kèm checkpoint để DRL không phải chép tay lần nữa.
            "class_names": CLASS_NAMES, "seg_label_lut": SEG_LABEL_LUT.tolist(),
            "roadline_id": ROADLINE_ID, "road_id": ROAD_ID, "sky_id": SKY_ID,
            "collect_fps": COLLECT_FPS, "control_dt": CONTROL_DT,
            "train_towns": TRAIN_TOWNS, "val_towns": VAL_TOWNS,
            "copycat_steer_mae": COPYCAT_STEER_MAE, "copycat_long_mae": COPYCAT_LONG_MAE,
            "seg_native_height": SEG_NATIVE_HEIGHT, "seg_native_width": SEG_NATIVE_WIDTH,
            "thin_cover_thresh": THIN_COVER_THRESH,
            "seg_checkpoint": os.path.basename(SEG_CHECKPOINT_PATH),
            "trained_on_predicted_seg": bool(USE_PREDICTED_SEGMENTATION),
            "scalar_feature_dim": SCALAR_FEATURE_DIM, "continuous_cols": CONTINUOUS_COLS,
            "raw_action_cols": RAW_ACTION_COLS, "scalar_feature_order": SCALAR_FEATURE_ORDER,
            "norm_stats": norm_stats, "traffic_light_vocab": TRAFFIC_LIGHT_VOCAB,
        }, STEER_CHECKPOINT_PATH)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"Dừng sớm ở epoch {epoch+1}"); break

print(f"Hoàn tất. Best Val Loss: {best_val_loss:.4f} | Tổng thời gian: {(time.time()-training_start)/60:.1f} phút | {STEER_CHECKPOINT_PATH}")


> Checkpoint lưu kèm `norm_stats`, `continuous_cols`, `raw_action_cols`,
> `scalar_feature_order` và `traffic_light_vocab` — bắt buộc nạp lại đúng các giá trị này lúc
> inference/demo, không tính lại từ dữ liệu mới. `drl_training/policy/observation.py` đọc
> trực tiếp các trường này từ checkpoint để dựng scalar vector đúng thứ tự khi warm-start
> actor PPO, thay vì hard-code lại — tránh hai bên (IL notebook trên Kaggle và DRL module
> chạy local) lệch nhau khi đặc trưng thay đổi sau này.


## 10. Biểu đồ huấn luyện

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(history["train_loss"], label="Train"); axes[0].plot(history["val_loss"], label="Val")
axes[0].set_title("Loss theo epoch"); axes[0].legend()
axes[1].plot(history["val_steer_mae"], label="Steer MAE"); axes[1].plot(history["val_long_mae"], label="Longitudinal MAE")
axes[1].set_title("MAE theo epoch (Val)"); axes[1].legend()
axes[2].plot(history["lr"], color="orange"); axes[2].set_title("LR schedule (Cosine)")
plt.tight_layout(); plt.show()
print(f"Thời gian trung bình/epoch: {np.mean(history['epoch_time']):.1f}s")


## 11. Đánh giá chi tiết trên tập Validation

Gồm: scatter dự đoán-thực tế, phân bố sai số, và **MAE theo từng trạng thái đèn tín hiệu**
(quan trọng để kiểm tra model có phản ứng đúng lúc đèn đỏ hay không — vì đây là nhãn hiếm).

In [ ]:
ckpt = torch.load(STEER_CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"]); model.eval()

all_preds, all_targets, all_aux = [], [], []
with torch.no_grad():
    for masks, scalars, aux, targets in val_loader:
        preds = model(masks.to(DEVICE), scalars.to(DEVICE)).cpu().numpy()
        all_preds.append(preds); all_targets.append(targets.numpy()); all_aux.append(aux.numpy())
all_preds = np.concatenate(all_preds); all_targets = np.concatenate(all_targets)
all_aux = np.concatenate(all_aux)  # cols: lane_offset_m, heading_error_rad, is_junction
all_tl = val_df["traffic_light_state"].values[:len(all_preds)]

labels = ["Steer", "Longitudinal"]
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for i, lb in enumerate(labels):
    axes[i].scatter(all_targets[:, i], all_preds[:, i], alpha=0.3, s=8)
    lims = [min(all_targets[:,i].min(), all_preds[:,i].min()), max(all_targets[:,i].max(), all_preds[:,i].max())]
    axes[i].plot(lims, lims, "r--", lw=1); axes[i].set_xlabel("Thực tế"); axes[i].set_ylabel("Dự đoán"); axes[i].set_title(lb)
plt.tight_layout(); plt.show()

errors = all_preds - all_targets
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(errors[:, 0], bins=50); axes[0].set_title("Phân bố sai số Steer")
axes[1].hist(errors[:, 1], bins=50); axes[1].set_title("Phân bố sai số Longitudinal")
plt.tight_layout(); plt.show()

mae_by_tl = pd.DataFrame({"tl": all_tl, "steer_ae": np.abs(errors[:, 0]), "long_ae": np.abs(errors[:, 1])})
mae_by_tl = mae_by_tl.groupby("tl")[["steer_ae", "long_ae"]].mean().reindex(TRAFFIC_LIGHT_VOCAB)
mae_by_tl.plot(kind="bar", figsize=(8, 4), title="MAE theo trạng thái đèn tín hiệu")
plt.tight_layout(); plt.show()
print(mae_by_tl)

# Sai số theo mức lệch làn (aux — KHÔNG đưa vào model, chỉ dùng để chẩn đoán ở đây).
# Kỳ vọng: steer_ae tăng dần khi |lane_offset_m| lớn — đây là vùng model phải lái mạnh để
# hồi phục, khó nhất, và cũng là vùng cần nhiều dữ liệu recovery/DAgger nhất nếu MAE cao.
lane_offset_bins = pd.cut(np.abs(all_aux[:, 0]), bins=[0, 0.15, 0.4, 0.8, np.inf],
                           labels=["<0.15m", "0.15-0.4m", "0.4-0.8m", ">0.8m"])
mae_by_offset = pd.DataFrame({"bin": lane_offset_bins, "steer_ae": np.abs(errors[:, 0])})
mae_by_offset = mae_by_offset.groupby("bin", observed=False)["steer_ae"].agg(["mean", "count"])
print("\nMAE steer theo |lane_offset_m| (chỉ để chẩn đoán, KHÔNG dùng làm input model):")
print(mae_by_offset)

mae_final = np.abs(errors).mean(axis=0)
print(f"\nMAE tổng — Steer: {mae_final[0]:.4f} | Longitudinal: {mae_final[1]:.4f}")

# [v3] Đối chiếu với baseline sao chép previous_steer (xem mục 4).
print("\n" + "=" * 58)
print(f"{'':<14}{'model':>10}{'baseline chép':>16}{'':>4}")
for _i, (_lb, _cc) in enumerate((("Steer", COPYCAT_STEER_MAE),
                                 ("Longitudinal", COPYCAT_LONG_MAE))):
    _gain = 100 * (1 - mae_final[_i] / _cc) if _cc > 0 else float("nan")
    _ok = "ĐẠT" if mae_final[_i] < _cc else "KHÔNG ĐẠT"
    print(f"{_lb:<14}{mae_final[_i]:>10.4f}{_cc:>16.4f}   {_gain:+6.1f}%  {_ok}")
print("=" * 58)
if mae_final[0] >= COPYCAT_STEER_MAE:
    print("⚠️  Model KHÔNG tốt hơn việc lặp lại lệnh lái của bước trước — nó đang bỏ qua")
    print("    ảnh segmentation. Đừng đưa checkpoint này sang DRL. Hướng xử lý:")
    print("    (1) kiểm tra corr(steer, previous_steer) ở mục 4, nếu > 0.98 thì FPS thu")
    print("        thập vẫn quá dày; (2) thử bỏ hẳn previous_* khỏi RAW_ACTION_COLS và")
    print("        train lại — mất một chút độ mượt nhưng buộc model học từ ảnh.")
else:
    print(f"✅ Model IL sẵn sàng tại: {STEER_CHECKPOINT_PATH}")
